In [ ]:
#instalare
#!pip install ultralytics

#yaml_content = """
# train: /content/dataset/images/train
# val: /content/dataset/images/val
# names: {0: 'fred', 1: 'daphne', 2: 'shaggy', 3: 'velma'}
# """
# with open("data.yaml", "w") as f:
#     f.write(yaml_content)

#antrenare
#from ultralytics import YOLO

# model = YOLO("yolov8n.pt")
# model.train(
#     data="data_task2.yaml",
#     epochs=20,
#     imgsz=640,
#     batch=16,
#     project="runs/detect",
#     name="task2_yolo_final",
#     mosaic=1.0,
#     val=True
# )

In [ ]:
from ultralytics import YOLO
import glob
import os
import numpy as np
from tqdm import tqdm
import ntpath

TEST_IMAGE_SOURCE = "../../../../evaluare/fake_test"
SOLUTION_SAVE_DIRECTORY = "output"

detection_model = YOLO("best_task2.pt")
print(f"starting run on {TEST_IMAGE_SOURCE}")

test_image_paths = sorted(glob.glob(os.path.join(TEST_IMAGE_SOURCE, "*.jpg")))

character_mapping = {0: 'fred', 1: 'daphne', 2: 'shaggy', 3: 'velma'}
final_results = {char: {'boxes': [], 'scores': [], 'names': []} for char in character_mapping.values()}

for image_path in tqdm(test_image_paths, desc="YOLO run"):
    inference_results = detection_model.predict(image_path, conf=0.001, iou=0.5, verbose=False)

    current_filename = ntpath.basename(image_path)
    for result in inference_results:
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        class_ids = result.boxes.cls.cpu().numpy().astype(int)

        for box, score, cls_id in zip(boxes_xyxy, scores, class_ids):
            if cls_id in character_mapping:
                char_name = character_mapping[cls_id]
                x1, y1, x2, y2 = map(int, box)
                final_results[char_name]['boxes'].append([x1, y1, x2, y2])
                final_results[char_name]['scores'].append(float(score))
                final_results[char_name]['names'].append(current_filename)

os.makedirs(SOLUTION_SAVE_DIRECTORY, exist_ok=True)

for char_name in character_mapping.values():
    np.save(os.path.join(SOLUTION_SAVE_DIRECTORY, f'detections_{char_name}.npy'), np.array(final_results[char_name]['boxes']))
    np.save(os.path.join(SOLUTION_SAVE_DIRECTORY, f'scores_{char_name}.npy'), np.array(final_results[char_name]['scores']))
    np.save(os.path.join(SOLUTION_SAVE_DIRECTORY, f'file_names_{char_name}.npy'), np.array(final_results[char_name]['names']))

print(f"saved detections to {SOLUTION_SAVE_DIRECTORY}")